In [2]:
pip install igraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 16.1 MB/s eta 0:00:00 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import TruncatedSVD
import igraph as ig  # pip install python-igraph
from collections import Counter
from node2vec import Node2Vec
import networkx as nx


In [2]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.10.0+cu128
12.8
True
NVIDIA GeForce RTX 5060


In [3]:
def remove_redun(el, verbose=False):
    if verbose:
        print("Original Size: ", len(el))

    el_new = el.iloc[:, 0:2].apply(sorted, axis=1)
    el_new = pd.DataFrame.from_dict(dict(zip(el_new.index, el_new.values))).T
    el_new = el_new.drop_duplicates()
    if verbose:
        postDrop = len(el_new)
        print("After Dropping Duplicates: ", len(el_new), "(-", len(el)-postDrop, ")")

    el_new = el_new.merge(el, left_on=[el_new.columns[0], el_new.columns[1]],
                          right_on=[el.columns[0], el.columns[1]])
    if verbose:
        print("After Merging: ", len(el_new), "(-", postDrop-len(el_new), ")")
        print()

    return el_new.iloc[:, 2:]


def map_IDs(el, gmap, verbose=False, dropNaNvalues=True):
    gp_map_f = gmap.set_index('#string_protein_id')['alias']
    el_converted = el.reset_index(drop=True)

    el_converted[el_converted.columns[0]] = el_converted[el_converted.columns[0]].map(gp_map_f)
    el_converted[el_converted.columns[1]] = el_converted[el_converted.columns[1]].map(gp_map_f)

    if verbose:
        print("NaN values per Column:",
              el_converted[el_converted.columns[0]].isna().sum(),
              el_converted[el_converted.columns[1]].isna().sum())

    if dropNaNvalues:
        el_converted = el_converted.dropna()
        if verbose:
            print("New edge list size:", len(el_converted), "( -", len(el)-len(el_converted), ")")

    return el_converted


def ImportSTRING():
    el_map = pd.read_csv(
        r"9606.protein.aliases.v12.0.txt",
        sep="\t"
    )
    el = pd.read_csv(
        r"9606.protein.links.v12.0_sc.txt",
        sep=" "
    )
    el_map = el_map.loc[el_map.source == 'Ensembl_gene']

    el = remove_redun(el, True)
    el = map_IDs(el, el_map, verbose=True)

    return el

In [35]:
def ImportDGN():
    dgn = pd.read_csv("Liver_Summary_GDA_ALL.csv")
    dgn_dict = pd.read_csv(
        "gda_dictionary.csv",
        index_col=None
    )

    score_threshold = 0.6 
    ei_threshold    = 0.5

    dgn = dgn[['gene', 'evidenceIndexGDA', 'scoreGDA']]
    dgn = dgn.loc[dgn['scoreGDA'] >= score_threshold]
    dgn = dgn.loc[dgn['evidenceIndexGDA'] > ei_threshold]
    dgn.rename({'scoreGDA': 'gda_score'}, axis=1, inplace=True)
    dgn = dgn.merge(dgn_dict, on="gene").drop(['gene'], axis=1)
    dgn['gda_score'] = 1

    return dgn[['ensembl', 'gda_score']]

In [5]:
def ImportHPA():
    hpa = pd.read_csv(
        r"liver.tsv",
        sep='\t'
    ).drop_duplicates(subset='Gene')

    identifiers = ["Gene", "Ensembl"]
    discrete_features = [
        "Protein class", "Biological process", "Molecular function",
        "Disease involvement", "Subcellular location",
    ]

    hpa_features = hpa.loc[:, hpa.columns.isin(identifiers + discrete_features)]

    # normalise continuous features

    def explode(feature):
        return feature.apply(lambda x: x.replace(' ', '').split(','))

    hpa_clean = hpa.fillna('')
    for ft in discrete_features:
        hpa_clean[ft] = explode(hpa_clean[ft])

    protein_class        = hpa_clean["Protein class"].explode().unique()
    biological_process   = hpa_clean["Biological process"].explode().unique()
    molecular_function   = hpa_clean["Molecular function"].explode().unique()
    disease_involvement  = hpa_clean["Disease involvement"].explode().unique()
    subcellular_location = hpa_clean["Subcellular location"].explode().unique()

    GO_features = np.concatenate([
        protein_class, biological_process, molecular_function,
        disease_involvement, subcellular_location
    ])

    RowFeatures = pd.DataFrame(data=0, index=hpa_clean['Ensembl'], columns=GO_features)
    counter = 0
    for index, row in RowFeatures.iterrows():
        features = hpa_clean.iloc[counter][
            ['Protein class', 'Biological process', 'Molecular function',
             'Disease involvement', 'Subcellular location']
        ].to_list()
        flattened = [item for sublist in features for item in sublist if item]
        for t in flattened:
            row[t] = 1
        counter += 1

    # truncated SVD to 200 dims
    n_comp    = 200
    svd       = TruncatedSVD(n_components=n_comp)
    svdModel  = svd.fit(RowFeatures)
    visits_emb = svdModel.transform(RowFeatures)

    hpa_reduced = pd.DataFrame(data=visits_emb, index=RowFeatures.index).reset_index(names="Ensembl")

    hpa_final = hpa_reduced

    hpa_final.columns = ['hpa_' + str(col) for col in hpa_final.columns]
    hpa_final = hpa_final.rename({
        'hpa_Ensembl': 'ensembl',
    }, axis=1)

    return hpa_final

In [16]:
def ImportGDC():
    gdc = pd.read_csv("CNVs LIHC_9_7_26.tsv", sep='\t')

    gdc = gdc.rename(columns={
        'ensembl': 'ensembl',
        'cohort_ssm_affected_cases_percentage': 'nih_ssm_in_cohort',
        'gdc_ssm_affected_cases_percentage':    'nih_ssm_across_gdc',
        'cohort_cnv_gain_cases_percentage':     'nih_cnv_gain',
        'num_mutations':                        'nih_tot_mutations',
    })

    gdc['nih_cnv_loss'] = gdc['cohort_cnv_homozygous_deletion_cases_percentage']

    for col in ['nih_ssm_in_cohort', 'nih_ssm_across_gdc', 'nih_cnv_gain', 'nih_cnv_loss']:
        gdc[col] = gdc[col] / 100.0

    gdc = gdc[['ensembl', 'nih_ssm_in_cohort', 'nih_ssm_across_gdc',
               'nih_cnv_gain', 'nih_cnv_loss', 'nih_tot_mutations']]

    return gdc

In [9]:
# --- node2vec network embeddings ---
import os

def Runnode2vec(filepath):
    df = pd.read_csv(filepath, sep="\t", header=None, names=["source", "target", "weight"])
    df["weight"] /= 1000 

    G = nx.from_pandas_edgelist(df, "source", "target", ["weight"], create_using=nx.Graph())


    n_workers = max(1, min(4, os.cpu_count() or 1))

    node2vec = Node2Vec(
        G, dimensions=128, walk_length=60, num_walks=15,
        workers=n_workers, p=1, q=0.5,
        quiet=True,          # skip per-walk progress bar overhead
    )
    model = node2vec.fit(window=10, min_count=1, batch_words=4)

    emd = pd.DataFrame([model.wv[str(node)] for node in G.nodes()], index=G.nodes())
    emd.columns = [f'network_{i}' for i in range(emd.shape[1])]

    return emd.reset_index().rename(columns={"index": "ensembl"})

## Run pipeline

In [10]:
hpa = ImportHPA()
hpa

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_190,hpa_191,hpa_192,hpa_193,hpa_194,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199
0,ENSG00000121410,0.840824,0.192866,0.256355,0.127673,-0.394840,0.631817,-0.722540,-0.199264,0.113550,...,0.003278,0.006103,0.010329,0.003375,-0.000418,0.008248,-0.001235,0.003150,0.008801,0.007741
1,ENSG00000148584,0.956038,0.718451,0.138687,0.064161,-0.180600,-0.364820,-0.358312,0.585720,-0.238726,...,-0.006925,-0.002353,-0.010268,-0.009177,-0.050242,0.075173,0.031971,0.146956,-0.093724,0.176523
2,ENSG00000175899,0.432260,-0.161568,-0.127495,0.096712,-0.266326,1.734092,-0.164837,0.383489,0.011611,...,-0.011188,0.054190,0.006284,-0.039820,0.015528,0.018060,-0.014972,-0.015421,0.016844,0.022413
3,ENSG00000128274,0.702846,-0.655898,1.082080,-0.442310,1.410061,-0.046312,0.195349,-0.025325,0.106249,...,0.005881,-0.023110,0.045645,0.033798,0.035691,0.070165,-0.063672,0.018423,0.040853,-0.015762
4,ENSG00000094914,2.404675,-0.800577,-0.251059,0.427358,-0.223315,-1.097957,0.371342,0.242683,0.711703,...,0.013807,0.001077,-0.023522,0.000989,-0.008937,0.036272,0.012650,-0.022177,-0.041297,0.023764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13530,ENSG00000203995,0.938609,0.678271,0.188220,0.015564,-0.081779,-0.364309,-0.220376,0.502737,-0.383299,...,-0.010082,-0.011358,0.007040,-0.005130,0.005363,0.007703,-0.008422,-0.006506,-0.001445,-0.016363
13531,ENSG00000162378,0.852371,0.274948,0.401533,0.291372,-0.101406,-0.008556,-0.458409,-0.375056,-0.668454,...,-0.000189,-0.012437,0.000825,0.001170,0.015384,0.015154,-0.011693,0.000034,-0.000372,-0.016885
13532,ENSG00000159840,0.955798,0.216098,0.351747,0.343809,-0.419039,0.642473,-0.466143,-0.260253,0.088280,...,-0.007840,-0.020512,0.019136,0.004160,0.014148,0.002650,-0.010753,-0.007253,-0.007571,0.000340
13533,ENSG00000074755,0.750615,0.742984,-0.588599,0.413192,1.179031,-0.121606,0.298645,0.337813,0.265360,...,-0.002896,0.002894,0.000189,0.002797,-0.004534,-0.007850,-0.003300,0.002808,0.014003,-0.002420


In [11]:
el = ImportSTRING()

print(f'Edges before filtering: {len(el):,}')
el = el.loc[el['combined_score'] > 700].reset_index(drop=True)
print(f'Edges after removing score >= 400: {len(el):,}')

el

Original Size:  13715404
After Dropping Duplicates:  6857702 (- 6857702 )
After Merging:  6857702 (- 0 )

NaN values per Column: 0 0
New edge list size: 6857702 ( - 0 )
Edges before filtering: 6,857,702
Edges after removing score >= 400: 236,000


,protein1,protein2,combined_score
0,ENSG00000004059,ENSG00000072818,825
1,ENSG00000004059,ENSG00000122218,718
2,ENSG00000004059,ENSG00000090565,952
3,ENSG00000004059,ENSG00000184432,752
4,ENSG00000004059,ENSG00000105669,795
...,...,...,...
235995,ENSG00000143933,ENSG00000070808,962
235996,ENSG00000143933,ENSG00000145335,918
235997,ENSG00000162434,ENSG00000051382,933
235998,ENSG00000070808,ENSG00000121281,707


In [17]:
gdc=ImportGDC()
gdc

,ensembl,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000155657,0.5081,0.2901,0.2172,0.0,825
1,ENSG00000141510,0.5027,0.3137,0.0489,0.0,208
2,ENSG00000181143,0.4436,0.1849,0.0333,0.0,505
3,ENSG00000198626,0.4204,0.1305,0.3933,0.0,434
4,ENSG00000164796,0.4132,0.1315,0.3014,0.0,409
...,...,...,...,...,...,...
20363,ENSG00000288660,0.0000,0.0000,0.2270,0.0,0
20364,ENSG00000288669,0.0000,0.0000,0.0313,0.0,0
20365,ENSG00000288671,0.0000,0.0000,0.1311,0.0,0
20366,ENSG00000288674,0.0000,0.0000,0.4031,0.0,0


In [21]:
master = hpa
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_190,hpa_191,hpa_192,hpa_193,hpa_194,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199
0,ENSG00000121410,0.840824,0.192866,0.256355,0.127673,-0.394840,0.631817,-0.722540,-0.199264,0.113550,...,0.003278,0.006103,0.010329,0.003375,-0.000418,0.008248,-0.001235,0.003150,0.008801,0.007741
1,ENSG00000148584,0.956038,0.718451,0.138687,0.064161,-0.180600,-0.364820,-0.358312,0.585720,-0.238726,...,-0.006925,-0.002353,-0.010268,-0.009177,-0.050242,0.075173,0.031971,0.146956,-0.093724,0.176523
2,ENSG00000175899,0.432260,-0.161568,-0.127495,0.096712,-0.266326,1.734092,-0.164837,0.383489,0.011611,...,-0.011188,0.054190,0.006284,-0.039820,0.015528,0.018060,-0.014972,-0.015421,0.016844,0.022413
3,ENSG00000128274,0.702846,-0.655898,1.082080,-0.442310,1.410061,-0.046312,0.195349,-0.025325,0.106249,...,0.005881,-0.023110,0.045645,0.033798,0.035691,0.070165,-0.063672,0.018423,0.040853,-0.015762
4,ENSG00000094914,2.404675,-0.800577,-0.251059,0.427358,-0.223315,-1.097957,0.371342,0.242683,0.711703,...,0.013807,0.001077,-0.023522,0.000989,-0.008937,0.036272,0.012650,-0.022177,-0.041297,0.023764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13530,ENSG00000203995,0.938609,0.678271,0.188220,0.015564,-0.081779,-0.364309,-0.220376,0.502737,-0.383299,...,-0.010082,-0.011358,0.007040,-0.005130,0.005363,0.007703,-0.008422,-0.006506,-0.001445,-0.016363
13531,ENSG00000162378,0.852371,0.274948,0.401533,0.291372,-0.101406,-0.008556,-0.458409,-0.375056,-0.668454,...,-0.000189,-0.012437,0.000825,0.001170,0.015384,0.015154,-0.011693,0.000034,-0.000372,-0.016885
13532,ENSG00000159840,0.955798,0.216098,0.351747,0.343809,-0.419039,0.642473,-0.466143,-0.260253,0.088280,...,-0.007840,-0.020512,0.019136,0.004160,0.014148,0.002650,-0.010753,-0.007253,-0.007571,0.000340
13533,ENSG00000074755,0.750615,0.742984,-0.588599,0.413192,1.179031,-0.121606,0.298645,0.337813,0.265360,...,-0.002896,0.002894,0.000189,0.002797,-0.004534,-0.007850,-0.003300,0.002808,0.014003,-0.002420


In [22]:
master = master.merge(gdc, on='ensembl')
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000121410,0.840824,0.192866,0.256355,0.127673,-0.394840,0.631817,-0.722540,-0.199264,0.113550,...,0.008248,-0.001235,0.003150,0.008801,0.007741,0.0107,0.0079,0.1155,0.0,6
1,ENSG00000148584,0.956038,0.718451,0.138687,0.064161,-0.180600,-0.364820,-0.358312,0.585720,-0.238726,...,0.075173,0.031971,0.146956,-0.093724,0.176523,0.0322,0.0166,0.1292,0.0,18
2,ENSG00000175899,0.432260,-0.161568,-0.127495,0.096712,-0.266326,1.734092,-0.164837,0.383489,0.011611,...,0.018060,-0.014972,-0.015421,0.016844,0.022413,0.0394,0.0247,0.1722,0.0,23
3,ENSG00000128274,0.702846,-0.655898,1.082080,-0.442310,1.410061,-0.046312,0.195349,-0.025325,0.106249,...,0.070165,-0.063672,0.018423,0.040853,-0.015762,0.0000,0.0066,0.0881,0.0,0
4,ENSG00000094914,2.404675,-0.800577,-0.251059,0.427358,-0.223315,-1.097957,0.371342,0.242683,0.711703,...,0.036272,0.012650,-0.022177,-0.041297,0.023764,0.0018,0.0072,0.1840,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13401,ENSG00000203995,0.938609,0.678271,0.188220,0.015564,-0.081779,-0.364309,-0.220376,0.502737,-0.383299,...,0.007703,-0.008422,-0.006506,-0.001445,-0.016363,0.0000,0.0031,0.1722,0.0,0
13402,ENSG00000162378,0.852371,0.274948,0.401533,0.291372,-0.101406,-0.008556,-0.458409,-0.375056,-0.668454,...,0.015154,-0.011693,0.000034,-0.000372,-0.016885,0.0197,0.0093,0.1722,0.0,11
13403,ENSG00000159840,0.955798,0.216098,0.351747,0.343809,-0.419039,0.642473,-0.466143,-0.260253,0.088280,...,0.002650,-0.010753,-0.007253,-0.007571,0.000340,0.0125,0.0080,0.2505,0.0,8
13404,ENSG00000074755,0.750615,0.742984,-0.588599,0.413192,1.179031,-0.121606,0.298645,0.337813,0.265360,...,-0.007850,-0.003300,0.002808,0.014003,-0.002420,0.0411,0.0260,0.0548,0.0,24


In [23]:
list_of_dropped_genes=pd.read_csv("18k_genes_LIHC_gene_x_celltype_mean_expression_with_ensembl.csv")
master_590 = master[
    master["ensembl"].isin(list_of_dropped_genes["ensembl"])
]
master=master_590
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000121410,0.840824,0.192866,0.256355,0.127673,-0.394840,0.631817,-0.722540,-0.199264,0.113550,...,0.008248,-0.001235,0.003150,0.008801,0.007741,0.0107,0.0079,0.1155,0.0,6
1,ENSG00000148584,0.956038,0.718451,0.138687,0.064161,-0.180600,-0.364820,-0.358312,0.585720,-0.238726,...,0.075173,0.031971,0.146956,-0.093724,0.176523,0.0322,0.0166,0.1292,0.0,18
2,ENSG00000175899,0.432260,-0.161568,-0.127495,0.096712,-0.266326,1.734092,-0.164837,0.383489,0.011611,...,0.018060,-0.014972,-0.015421,0.016844,0.022413,0.0394,0.0247,0.1722,0.0,23
3,ENSG00000128274,0.702846,-0.655898,1.082080,-0.442310,1.410061,-0.046312,0.195349,-0.025325,0.106249,...,0.070165,-0.063672,0.018423,0.040853,-0.015762,0.0000,0.0066,0.0881,0.0,0
4,ENSG00000094914,2.404675,-0.800577,-0.251059,0.427358,-0.223315,-1.097957,0.371342,0.242683,0.711703,...,0.036272,0.012650,-0.022177,-0.041297,0.023764,0.0018,0.0072,0.1840,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13401,ENSG00000203995,0.938609,0.678271,0.188220,0.015564,-0.081779,-0.364309,-0.220376,0.502737,-0.383299,...,0.007703,-0.008422,-0.006506,-0.001445,-0.016363,0.0000,0.0031,0.1722,0.0,0
13402,ENSG00000162378,0.852371,0.274948,0.401533,0.291372,-0.101406,-0.008556,-0.458409,-0.375056,-0.668454,...,0.015154,-0.011693,0.000034,-0.000372,-0.016885,0.0197,0.0093,0.1722,0.0,11
13403,ENSG00000159840,0.955798,0.216098,0.351747,0.343809,-0.419039,0.642473,-0.466143,-0.260253,0.088280,...,0.002650,-0.010753,-0.007253,-0.007571,0.000340,0.0125,0.0080,0.2505,0.0,8
13404,ENSG00000074755,0.750615,0.742984,-0.588599,0.413192,1.179031,-0.121606,0.298645,0.337813,0.265360,...,-0.007850,-0.003300,0.002808,0.014003,-0.002420,0.0411,0.0260,0.0548,0.0,24


In [24]:
master = master.reset_index(drop=True)
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000121410,0.840824,0.192866,0.256355,0.127673,-0.394840,0.631817,-0.722540,-0.199264,0.113550,...,0.008248,-0.001235,0.003150,0.008801,0.007741,0.0107,0.0079,0.1155,0.0,6
1,ENSG00000148584,0.956038,0.718451,0.138687,0.064161,-0.180600,-0.364820,-0.358312,0.585720,-0.238726,...,0.075173,0.031971,0.146956,-0.093724,0.176523,0.0322,0.0166,0.1292,0.0,18
2,ENSG00000175899,0.432260,-0.161568,-0.127495,0.096712,-0.266326,1.734092,-0.164837,0.383489,0.011611,...,0.018060,-0.014972,-0.015421,0.016844,0.022413,0.0394,0.0247,0.1722,0.0,23
3,ENSG00000128274,0.702846,-0.655898,1.082080,-0.442310,1.410061,-0.046312,0.195349,-0.025325,0.106249,...,0.070165,-0.063672,0.018423,0.040853,-0.015762,0.0000,0.0066,0.0881,0.0,0
4,ENSG00000094914,2.404675,-0.800577,-0.251059,0.427358,-0.223315,-1.097957,0.371342,0.242683,0.711703,...,0.036272,0.012650,-0.022177,-0.041297,0.023764,0.0018,0.0072,0.1840,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12393,ENSG00000203995,0.938609,0.678271,0.188220,0.015564,-0.081779,-0.364309,-0.220376,0.502737,-0.383299,...,0.007703,-0.008422,-0.006506,-0.001445,-0.016363,0.0000,0.0031,0.1722,0.0,0
12394,ENSG00000162378,0.852371,0.274948,0.401533,0.291372,-0.101406,-0.008556,-0.458409,-0.375056,-0.668454,...,0.015154,-0.011693,0.000034,-0.000372,-0.016885,0.0197,0.0093,0.1722,0.0,11
12395,ENSG00000159840,0.955798,0.216098,0.351747,0.343809,-0.419039,0.642473,-0.466143,-0.260253,0.088280,...,0.002650,-0.010753,-0.007253,-0.007571,0.000340,0.0125,0.0080,0.2505,0.0,8
12396,ENSG00000074755,0.750615,0.742984,-0.588599,0.413192,1.179031,-0.121606,0.298645,0.337813,0.265360,...,-0.007850,-0.003300,0.002808,0.014003,-0.002420,0.0411,0.0260,0.0548,0.0,24


In [25]:
el_allgenes = pd.concat([el['protein1'], el['protein2']]).drop_duplicates()
master = master.loc[master['ensembl'].isin(el_allgenes)]
print(f'Genes after STRING intersection: {len(master)}')
master

Genes after STRING intersection: 10980


,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,hpa_195,hpa_196,hpa_197,hpa_198,hpa_199,nih_ssm_in_cohort,nih_ssm_across_gdc,nih_cnv_gain,nih_cnv_loss,nih_tot_mutations
0,ENSG00000121410,0.840824,0.192866,0.256355,0.127673,-0.394840,0.631817,-0.722540,-0.199264,0.113550,...,0.008248,-0.001235,0.003150,0.008801,0.007741,0.0107,0.0079,0.1155,0.0,6
1,ENSG00000148584,0.956038,0.718451,0.138687,0.064161,-0.180600,-0.364820,-0.358312,0.585720,-0.238726,...,0.075173,0.031971,0.146956,-0.093724,0.176523,0.0322,0.0166,0.1292,0.0,18
2,ENSG00000175899,0.432260,-0.161568,-0.127495,0.096712,-0.266326,1.734092,-0.164837,0.383489,0.011611,...,0.018060,-0.014972,-0.015421,0.016844,0.022413,0.0394,0.0247,0.1722,0.0,23
3,ENSG00000128274,0.702846,-0.655898,1.082080,-0.442310,1.410061,-0.046312,0.195349,-0.025325,0.106249,...,0.070165,-0.063672,0.018423,0.040853,-0.015762,0.0000,0.0066,0.0881,0.0,0
4,ENSG00000094914,2.404675,-0.800577,-0.251059,0.427358,-0.223315,-1.097957,0.371342,0.242683,0.711703,...,0.036272,0.012650,-0.022177,-0.041297,0.023764,0.0018,0.0072,0.1840,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12392,ENSG00000070476,0.975150,1.108658,-0.669627,-0.080018,0.599901,0.058349,-0.214723,-0.869283,0.073015,...,-0.003560,-0.008310,0.010812,0.001621,-0.009347,0.0143,0.0101,0.1898,0.0,10
12393,ENSG00000203995,0.938609,0.678271,0.188220,0.015564,-0.081779,-0.364309,-0.220376,0.502737,-0.383299,...,0.007703,-0.008422,-0.006506,-0.001445,-0.016363,0.0000,0.0031,0.1722,0.0,0
12394,ENSG00000162378,0.852371,0.274948,0.401533,0.291372,-0.101406,-0.008556,-0.458409,-0.375056,-0.668454,...,0.015154,-0.011693,0.000034,-0.000372,-0.016885,0.0197,0.0093,0.1722,0.0,11
12395,ENSG00000159840,0.955798,0.216098,0.351747,0.343809,-0.419039,0.642473,-0.466143,-0.260253,0.088280,...,0.002650,-0.010753,-0.007253,-0.007571,0.000340,0.0125,0.0080,0.2505,0.0,8


In [26]:
el_intersect = (
    el.iloc[:, :3]
    .merge(master["ensembl"], right_on="ensembl", left_on='protein1')
    .drop("ensembl", axis=1)
)
el_intersect = (
    el_intersect
    .merge(master["ensembl"], right_on="ensembl", left_on='protein2')
    .drop("ensembl", axis=1)
    .rename(columns={'protein1': 'gene1', 'protein2': 'gene2'})
)

el = el_intersect.merge(el, right_on=['protein1', 'protein2'], left_on=['gene1', 'gene2']).drop(['protein1', 'protein2'], axis=1)
el

,gene1,gene2,combined_score_x,combined_score_y
0,ENSG00000004059,ENSG00000072818,825,825
1,ENSG00000004059,ENSG00000122218,718,718
2,ENSG00000004059,ENSG00000090565,952,952
3,ENSG00000004059,ENSG00000184432,752,752
4,ENSG00000004059,ENSG00000105669,795,795
...,...,...,...,...
155003,ENSG00000198668,ENSG00000145335,973,973
155004,ENSG00000274211,ENSG00000162434,786,786
155005,ENSG00000143933,ENSG00000145335,918,918
155006,ENSG00000162434,ENSG00000051382,933,933


In [27]:
el[['gene1', 'gene2', 'combined_score_x']].to_csv(
    '18k_lihc.edg', index=False, header=False, sep='\t'
)

In [28]:
df = pd.read_csv(
    "18k_lihc.edg",
    sep="\t", header=None, names=["source", "target", "weight"]
)
print(df.head())
print('NaN weights:', df["weight"].isna().sum())

            source           target  weight
0  ENSG00000004059  ENSG00000072818     825
1  ENSG00000004059  ENSG00000122218     718
2  ENSG00000004059  ENSG00000090565     952
3  ENSG00000004059  ENSG00000184432     752
4  ENSG00000004059  ENSG00000105669     795
NaN weights: 0


In [29]:
network = Runnode2vec("18k_lihc.edg")
master = master.merge(network, on='ensembl')
master.to_csv("node_node2vec_data_latest.csv", index=None)
master

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,network_118,network_119,network_120,network_121,network_122,network_123,network_124,network_125,network_126,network_127
0,ENSG00000121410,0.840824,0.192866,0.256355,0.127673,-0.394840,0.631817,-0.722540,-0.199264,0.113550,...,0.025941,-0.048900,-0.462306,-0.424690,0.353269,0.015792,-0.097789,-0.139160,0.182371,-0.159256
1,ENSG00000148584,0.956038,0.718451,0.138687,0.064161,-0.180600,-0.364820,-0.358312,0.585720,-0.238726,...,0.611819,0.168825,0.392611,-0.388210,0.215283,0.197297,-0.076665,0.151132,-0.346299,-0.375968
2,ENSG00000175899,0.432260,-0.161568,-0.127495,0.096712,-0.266326,1.734092,-0.164837,0.383489,0.011611,...,0.551872,-0.433762,-0.236415,-0.255862,0.443648,0.454124,0.033165,-0.031726,-0.223358,-0.275842
3,ENSG00000128274,0.702846,-0.655898,1.082080,-0.442310,1.410061,-0.046312,0.195349,-0.025325,0.106249,...,0.446977,-0.375446,0.116082,0.141657,0.746456,-0.021312,-0.059460,-0.444979,-0.365417,-0.512251
4,ENSG00000094914,2.404675,-0.800577,-0.251059,0.427358,-0.223315,-1.097957,0.371342,0.242683,0.711703,...,0.028497,-0.016668,-0.374651,0.012062,0.138791,0.105280,-0.000671,0.157839,0.075712,0.473551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10770,ENSG00000070476,0.975150,1.108658,-0.669627,-0.080018,0.599901,0.058349,-0.214723,-0.869283,0.073015,...,0.191761,-0.176635,-0.709926,-0.067333,0.370018,-0.324991,-0.353321,-0.193468,-0.138144,-0.274419
10771,ENSG00000203995,0.938609,0.678271,0.188220,0.015564,-0.081779,-0.364309,-0.220376,0.502737,-0.383299,...,0.112402,-0.070429,0.224809,0.200839,0.391155,-0.137069,-0.135122,0.413285,-0.035269,0.021615
10772,ENSG00000162378,0.852371,0.274948,0.401533,0.291372,-0.101406,-0.008556,-0.458409,-0.375056,-0.668454,...,0.299806,0.519610,0.400972,0.113265,0.214152,-0.129978,-0.277832,-0.089681,-0.479916,-0.298233
10773,ENSG00000159840,0.955798,0.216098,0.351747,0.343809,-0.419039,0.642473,-0.466143,-0.260253,0.088280,...,0.068800,0.018404,-0.017999,0.005955,0.199714,-0.150262,0.093214,-0.023515,0.076232,0.768893


In [30]:
master.to_csv("node_networkfeatures_18k_cnv.csv", index=False)

In [31]:
df = pd.read_csv(
    "18k_lihc.edg",
    sep="\t", header=None, names=["source", "target", "weight"]
)
print(df.head())
print('NaN weights:', df["weight"].isna().sum())

            source           target  weight
0  ENSG00000004059  ENSG00000072818     825
1  ENSG00000004059  ENSG00000122218     718
2  ENSG00000004059  ENSG00000090565     952
3  ENSG00000004059  ENSG00000184432     752
4  ENSG00000004059  ENSG00000105669     795
NaN weights: 0


In [32]:
edge_array = df[["source", "target", "weight"]].to_numpy()
np.save("18k_lihc.npy", edge_array, allow_pickle=True)
print("Saved edge_list_latest1.npy with shape:", edge_array.shape)

loaded = np.load("18k_lihc.npy", allow_pickle=True)
print(loaded[:5])

Saved edge_list_latest1.npy with shape: (155008, 3)
[['ENSG00000004059' 'ENSG00000072818' 825]
 ['ENSG00000004059' 'ENSG00000122218' 718]
 ['ENSG00000004059' 'ENSG00000090565' 952]
 ['ENSG00000004059' 'ENSG00000184432' 752]
 ['ENSG00000004059' 'ENSG00000105669' 795]]


In [33]:
master=pd.read_csv("node_networkfeatures_18k_cnv.csv")
master

,ensembl,hpa_0,hpa_1,hpa_2,hpa_3,hpa_4,hpa_5,hpa_6,hpa_7,hpa_8,...,network_118,network_119,network_120,network_121,network_122,network_123,network_124,network_125,network_126,network_127
0,ENSG00000121410,0.840824,0.192866,0.256355,0.127673,-0.394840,0.631817,-0.722540,-0.199264,0.113550,...,0.025941,-0.048900,-0.462306,-0.424690,0.353269,0.015792,-0.097789,-0.139160,0.182371,-0.159256
1,ENSG00000148584,0.956038,0.718451,0.138687,0.064161,-0.180600,-0.364820,-0.358312,0.585720,-0.238726,...,0.611819,0.168825,0.392611,-0.388210,0.215283,0.197297,-0.076665,0.151132,-0.346299,-0.375968
2,ENSG00000175899,0.432260,-0.161568,-0.127495,0.096712,-0.266326,1.734092,-0.164837,0.383489,0.011611,...,0.551872,-0.433762,-0.236415,-0.255862,0.443648,0.454124,0.033165,-0.031726,-0.223358,-0.275842
3,ENSG00000128274,0.702846,-0.655898,1.082080,-0.442310,1.410061,-0.046312,0.195349,-0.025325,0.106249,...,0.446977,-0.375446,0.116082,0.141657,0.746456,-0.021312,-0.059460,-0.444979,-0.365417,-0.512251
4,ENSG00000094914,2.404675,-0.800577,-0.251059,0.427358,-0.223315,-1.097957,0.371342,0.242683,0.711703,...,0.028497,-0.016668,-0.374652,0.012062,0.138791,0.105280,-0.000671,0.157839,0.075712,0.473551
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10770,ENSG00000070476,0.975150,1.108658,-0.669627,-0.080018,0.599901,0.058349,-0.214723,-0.869283,0.073015,...,0.191761,-0.176635,-0.709926,-0.067333,0.370018,-0.324991,-0.353321,-0.193468,-0.138144,-0.274419
10771,ENSG00000203995,0.938609,0.678271,0.188220,0.015564,-0.081779,-0.364309,-0.220376,0.502737,-0.383299,...,0.112402,-0.070429,0.224809,0.200839,0.391155,-0.137069,-0.135122,0.413285,-0.035269,0.021615
10772,ENSG00000162378,0.852371,0.274948,0.401533,0.291372,-0.101406,-0.008556,-0.458409,-0.375056,-0.668454,...,0.299806,0.519610,0.400972,0.113265,0.214152,-0.129978,-0.277832,-0.089681,-0.479916,-0.298233
10773,ENSG00000159840,0.955798,0.216098,0.351747,0.343809,-0.419039,0.642473,-0.466143,-0.260253,0.088280,...,0.068800,0.018404,-0.017999,0.005955,0.199714,-0.150262,0.093214,-0.023515,0.076232,0.768893


In [36]:
dgn = ImportDGN()
dgn = dgn.loc[dgn['ensembl'].isin(master['ensembl'])]
print(f'LIHC GDA genes overlapping final master: {len(dgn)}')
dgn

LIHC GDA genes overlapping final master: 251


,ensembl,gda_score
0,ENSG00000141510,1
1,ENSG00000121879,1
2,ENSG00000168036,1
4,ENSG00000073756,1
5,ENSG00000136997,1
...,...,...
318,ENSG00000111371,1
319,ENSG00000121067,1
320,ENSG00000057657,1
323,ENSG00000104320,1


In [37]:
master["gda_score"] = np.nan
master.loc[master["ensembl"].isin(dgn["ensembl"]), "gda_score"] = 1

num_ones = (master["gda_score"] == 1).sum()
print("Number of positive (gda_score=1) genes:", num_ones)

Number of positive (gda_score=1) genes: 223


In [38]:
master.to_csv("200_18k_cnv.csv", index=None)
print('Saved: 200_node_network_features_latest.csv')

Saved: 200_node_network_features_latest.csv
